In [1]:
# =========================================================
# AI SOFTWARE DEVELOPMENT TEAM
# LANGGRAPH + LANGCHAIN + GROQ
# PRODUCTION-STYLE MULTI-AGENT SYSTEM
# =========================================================

# =========================================================
# INSTALL
# =========================================================

# !pip install -q langgraph langchain langchain-groq python-dotenv rich


# =========================================================
# IMPORTS
# =========================================================

from rich import print

import os
from dotenv import load_dotenv

from typing import TypedDict, Optional, Literal

from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

from langchain.agents import create_agent

from langgraph.graph import StateGraph, END


# =========================================================
# ENV VARIABLES
# =========================================================

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Optional
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# Optional
os.environ["LANGCHAIN_PROJECT"] = "multi-agent-dev-team"


# =========================================================
# LLM
# =========================================================

llm = init_chat_model(
    model="groq:llama-3.3-70b-versatile",
    temperature=0
)


# =========================================================
# WORKFLOW STATE
# =========================================================

class WorkflowState(TypedDict):

    # user request
    user_request: str

    # research outputs
    research_notes: list[str]

    # architecture
    architecture_plan: Optional[str]

    # code
    generated_code: Optional[str]

    # review
    review_feedback: Optional[str]

    review_approved: bool

    # tests
    test_results: Optional[str]

    # routing
    next_step: Optional[str]

    # retries
    retry_count: int

    max_retries: int

    # metadata
    current_agent: Optional[str]

    execution_logs: list[str]

    errors: list[str]

    status: str


# =========================================================
# TOOLS
# =========================================================

# ---------------------------------------------------------
# RESEARCH TOOL
# ---------------------------------------------------------

@tool
def search_docs(query: str) -> str:
    """
    Search technical documentation.
    """

    docs = {
        "oauth": """
OAuth Best Practices:
- Use secure redirect URIs
- Store tokens securely
- Use CSRF protection
- Use HTTPS only
- Use short-lived tokens
- Use Authlib for FastAPI
        """,

        "fastapi oauth": """
FastAPI OAuth Example:
- Use Authlib
- Configure callback routes
- Use session middleware
- Store secrets in env variables
        """,

        "csrf": """
CSRF Protection:
- Use state parameter
- Validate callback state
- Use signed sessions
        """
    }

    query = query.lower()

    results = []

    for key, value in docs.items():
        if key in query:
            results.append(value)

    if results:
        return "\n".join(results)

    return "No documentation found."


# ---------------------------------------------------------
# ARCHITECTURE TOOL
# ---------------------------------------------------------

@tool
def generate_architecture(requirement: str) -> str:
    """
    Generate architecture plan.
    """

    return f"""
Architecture Plan

Requirement:
{requirement}

System Components:

1. OAuth Login Routes
2. Session Middleware
3. Google OAuth Callback
4. CSRF State Validation
5. Token Storage Layer
6. Secure Secret Management
7. Error Handling
8. Authentication Middleware

Recommended Stack:
- FastAPI
- Authlib
- Starlette Sessions
"""


# ---------------------------------------------------------
# PACKAGE LOOKUP TOOL
# ---------------------------------------------------------

@tool
def lookup_package(package_name: str) -> str:
    """
    Lookup package information.
    """

    packages = {
        "authlib": """
Authlib:
- Supports OAuth2
- Works with FastAPI
- Recommended for OAuth integrations
        """,

        "fastapi": """
FastAPI:
- Async framework
- High performance
- Excellent auth ecosystem
        """
    }

    return packages.get(
        package_name.lower(),
        "Package not found."
    )


# ---------------------------------------------------------
# CODE GENERATION TOOL
# ---------------------------------------------------------

@tool
def write_code(requirement: str) -> str:
    """
    Generate implementation code.
    """

    return """
from fastapi import FastAPI, Request
from starlette.middleware.sessions import SessionMiddleware

from authlib.integrations.starlette_client import OAuth

app = FastAPI()

app.add_middleware(
    SessionMiddleware,
    secret_key="SUPER_SECRET"
)

oauth = OAuth()

oauth.register(
    name="google",
    client_id="GOOGLE_CLIENT_ID",
    client_secret="GOOGLE_SECRET",
    server_metadata_url=
    "https://accounts.google.com/.well-known/openid-configuration",
    client_kwargs={
        "scope": "openid email profile"
    }
)

@app.get("/login/google")
async def login_google(request: Request):

    redirect_uri = request.url_for("auth_callback")

    state = "csrf_protection_enabled"

    return await oauth.google.authorize_redirect(
        request,
        redirect_uri,
        state=state
    )


@app.get("/auth/callback")
async def auth_callback(request: Request):

    token = await oauth.google.authorize_access_token(
        request
    )

    user = token.get("userinfo")

    return {
        "user": user
    }
"""


# ---------------------------------------------------------
# REVIEW TOOL
# ---------------------------------------------------------

@tool
def review_code_tool(code: str) -> str:
    """
    Review generated code.
    """

    issues = []

    if "csrf" not in code.lower():
        issues.append("Missing CSRF protection.")

    if "secret_key" not in code.lower():
        issues.append("Missing session secret key.")

    if not issues:
        return """
APPROVED

Code looks good.
Security protections detected.
"""

    return f"""
REJECTED

Issues:
{issues}
"""


# ---------------------------------------------------------
# TEST TOOL
# ---------------------------------------------------------

@tool
def run_tests(code: str) -> str:
    """
    Simulate test execution.
    """

    if "FastAPI" in code and "authorize_redirect" in code:

        return """
TEST RESULTS

✅ OAuth route exists
✅ Callback route exists
✅ Session middleware detected
✅ OAuth redirect works

ALL TESTS PASSED
"""

    return """
TEST RESULTS

❌ Tests failed
"""


# =========================================================
# SYSTEM PROMPTS
# =========================================================

research_prompt = """
You are a Senior Software Research Engineer.

Responsibilities:
- Research APIs
- Research best practices
- Find security concerns
- Find package recommendations

Rules:
- ALWAYS use tools first
- NEVER hallucinate documentation
- Return concise technical notes
"""


architect_prompt = """
You are a Senior Software Architect.

Responsibilities:
- Design clean architectures
- Create implementation plans
- Think about scalability/security

Rules:
- Use tools
- Focus on maintainability
"""


coding_prompt = """
You are a Senior Backend Engineer.

Responsibilities:
- Generate production-grade code
- Implement security protections
- Follow best practices

Rules:
- ALWAYS use tools
- Include CSRF protection
- Use clean architecture
"""


review_prompt = """
You are a Principal Software Reviewer.

Responsibilities:
- Detect security flaws
- Detect missing protections
- Detect bad coding practices

Rules:
- Be strict
- Reject insecure code
"""


testing_prompt = """
You are a Senior QA Engineer.

Responsibilities:
- Validate generated code
- Detect failures
- Produce concise test reports
"""


# =========================================================
# AGENTS
# =========================================================

research_agent = create_agent(
    model=llm,
    tools=[search_docs],
    system_prompt=research_prompt
)

architect_agent = create_agent(
    model=llm,
    tools=[generate_architecture],
    system_prompt=architect_prompt
)

coding_agent = create_agent(
    model=llm,
    tools=[lookup_package, write_code],
    system_prompt=coding_prompt
)

review_agent = create_agent(
    model=llm,
    tools=[review_code_tool],
    system_prompt=review_prompt
)

testing_agent = create_agent(
    model=llm,
    tools=[run_tests],
    system_prompt=testing_prompt
)


# =========================================================
# STRUCTURED OUTPUTS
# =========================================================

class SupervisorDecision(BaseModel):

    next_step: Literal[
        "research",
        "architect",
        "coder",
        "review",
        "tester",
        "human_review",
        "finish"
    ]

    reasoning: str = Field(
        description="Reason for routing decision"
    )


# =========================================================
# SUPERVISOR NODE
# =========================================================

def supervisor_node(state: WorkflowState):

    print("\n")
    print("=" * 60)
    print("SUPERVISOR")
    print("=" * 60)

    structured_llm = llm.with_structured_output(
        SupervisorDecision
    )

    decision = structured_llm.invoke(f"""
You are an AI Engineering Manager coordinating
a software development workflow.

Current Workflow State:

User Request:
{state["user_request"]}

Research Notes:
{state["research_notes"]}

Architecture Plan:
{state["architecture_plan"]}

Generated Code:
{state["generated_code"]}

Review Feedback:
{state["review_feedback"]}

Review Approved:
{state["review_approved"]}

Test Results:
{state["test_results"]}

Retry Count:
{state["retry_count"]}

Max Retries:
{state["max_retries"]}

Rules:

1. Start with research
2. Then architecture
3. Then coding
4. Then review
5. If approved -> testing
6. If rejected -> coding retry
7. If retries exceeded -> human_review
8. If tests pass -> finish
""")

    next_step = decision.next_step

    # safety override
    if (
        state["retry_count"] >= state["max_retries"]
        and not state["review_approved"]
    ):
        next_step = "human_review"

    print(f"\nNext Step: {next_step}")
    print(f"Reasoning: {decision.reasoning}")

    return {
        "next_step": next_step,

        "execution_logs":
            state["execution_logs"]
            + [f"Supervisor routed to {next_step}"],

        "current_agent": "supervisor"
    }


# =========================================================
# RESEARCH NODE
# =========================================================

def research_node(state: WorkflowState):

    print("\n")
    print("=" * 60)
    print("RESEARCH AGENT")
    print("=" * 60)

    try:

        result = research_agent.invoke({
            "messages": [
                HumanMessage(
                    content=f"""
Research this software requirement.

Requirement:
{state["user_request"]}

IMPORTANT:
- You MUST use tools
- NEVER answer from memory
"""
                )
            ]
        })

        print("\nFULL MESSAGE TRACE:\n")

        for msg in result["messages"]:
            print(type(msg).__name__)
            print(msg)
            print()

        output = result["messages"][-1].content

        return {
            "research_notes":
                state["research_notes"] + [output],

            "execution_logs":
                state["execution_logs"]
                + ["Research completed"],

            "current_agent": "research_agent"
        }

    except Exception as e:

        return {
            "errors":
                state["errors"] + [str(e)],

            "status": "FAILED"
        }


# =========================================================
# ARCHITECT NODE
# =========================================================

def architect_node(state: WorkflowState):

    print("\n")
    print("=" * 60)
    print("ARCHITECT AGENT")
    print("=" * 60)

    result = architect_agent.invoke({
        "messages": [
            HumanMessage(
                content=f"""
Create architecture plan.

Requirement:
{state["user_request"]}

Research Notes:
{state["research_notes"]}

You MUST use tools.
"""
            )
        ]
    })

    output = result["messages"][-1].content

    print(output)

    return {
        "architecture_plan": output,

        "execution_logs":
            state["execution_logs"]
            + ["Architecture completed"],

        "current_agent": "architect_agent"
    }


# =========================================================
# CODING NODE
# =========================================================

def coding_node(state: WorkflowState):

    print("\n")
    print("=" * 60)
    print("CODING AGENT")
    print("=" * 60)

    result = coding_agent.invoke({
        "messages": [
            HumanMessage(
                content=f"""
Generate production-ready implementation.

Requirement:
{state["user_request"]}

Architecture:
{state["architecture_plan"]}

Research:
{state["research_notes"]}

IMPORTANT:
- Use tools
- Include CSRF protection
- Use best practices
"""
            )
        ]
    })

    output = result["messages"][-1].content

    print(output)

    return {
        "generated_code": output,

        "review_feedback": None,

        "review_approved": False,

        "retry_count": state["retry_count"] + 1,

        "execution_logs":
            state["execution_logs"]
            + ["Code generation completed"],

        "current_agent": "coding_agent"
    }


# =========================================================
# REVIEW NODE
# =========================================================

def review_node(state: WorkflowState):

    print("\n")
    print("=" * 60)
    print("REVIEW AGENT")
    print("=" * 60)

    result = review_agent.invoke({
        "messages": [
            HumanMessage(
                content=f"""
Review this generated code carefully.

Code:
{state["generated_code"]}

You MUST use tools.
"""
            )
        ]
    })

    output = result["messages"][-1].content

    print(output)

    approved = "APPROVED" in output

    return {
        "review_feedback": output,

        "review_approved": approved,

        "execution_logs":
            state["execution_logs"]
            + ["Review completed"],

        "current_agent": "review_agent"
    }


# =========================================================
# TESTING NODE
# =========================================================

def testing_node(state: WorkflowState):

    print("\n")
    print("=" * 60)
    print("TESTING AGENT")
    print("=" * 60)

    result = testing_agent.invoke({
        "messages": [
            HumanMessage(
                content=f"""
Run tests on this implementation.

Code:
{state["generated_code"]}

You MUST use tools.
"""
            )
        ]
    })

    output = result["messages"][-1].content

    print(output)

    return {
        "test_results": output,

        "status": "SUCCESS",

        "execution_logs":
            state["execution_logs"]
            + ["Testing completed"],

        "current_agent": "testing_agent"
    }


# =========================================================
# HUMAN REVIEW NODE
# =========================================================

def human_review_node(state: WorkflowState):

    print("\n")
    print("=" * 60)
    print("HUMAN REVIEW")
    print("=" * 60)

    print("\nReview Feedback:\n")
    print(state["review_feedback"])

    approval = input(
        "\nApprove manually? (yes/no): "
    )

    approved = approval.lower() == "yes"

    return {
        "review_approved": approved,

        "execution_logs":
            state["execution_logs"]
            + ["Human review completed"],

        "current_agent": "human_reviewer"
    }


# =========================================================
# ROUTER
# =========================================================

def supervisor_router(state: WorkflowState):

    return state["next_step"]


# =========================================================
# BUILD GRAPH
# =========================================================

graph = StateGraph(WorkflowState)


# =========================================================
# ADD NODES
# =========================================================

graph.add_node(
    "supervisor",
    supervisor_node
)

graph.add_node(
    "research",
    research_node
)

graph.add_node(
    "architect",
    architect_node
)

graph.add_node(
    "coder",
    coding_node
)

graph.add_node(
    "review",
    review_node
)

graph.add_node(
    "tester",
    testing_node
)

graph.add_node(
    "human_review",
    human_review_node
)


# =========================================================
# ENTRY POINT
# =========================================================

graph.set_entry_point("supervisor")


# =========================================================
# CONDITIONAL ROUTING
# =========================================================

graph.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "research": "research",

        "architect": "architect",

        "coder": "coder",

        "review": "review",

        "tester": "tester",

        "human_review": "human_review",

        "finish": END
    }
)


# =========================================================
# RETURN EDGES
# =========================================================

graph.add_edge(
    "research",
    "supervisor"
)

graph.add_edge(
    "architect",
    "supervisor"
)

graph.add_edge(
    "coder",
    "supervisor"
)

graph.add_edge(
    "review",
    "supervisor"
)

graph.add_edge(
    "tester",
    "supervisor"
)

graph.add_edge(
    "human_review",
    "supervisor"
)


# =========================================================
# COMPILE
# =========================================================

app = graph.compile()


# =========================================================
# INITIAL STATE
# =========================================================

initial_state: WorkflowState = {

    "user_request": """
Build a FastAPI Google OAuth login system
with secure authentication and CSRF protection.
""",

    "research_notes": [],

    "architecture_plan": None,

    "generated_code": None,

    "review_feedback": None,

    "review_approved": False,

    "test_results": None,

    "next_step": None,

    "retry_count": 0,

    "max_retries": 3,

    "current_agent": None,

    "execution_logs": [],

    "errors": [],

    "status": "RUNNING"
}


# =========================================================
# EXECUTE WORKFLOW
# =========================================================

final_state = app.invoke(initial_state)


# =========================================================
# FINAL OUTPUT
# =========================================================

print("\n")
print("=" * 60)
print("FINAL STATE")
print("=" * 60)

for key, value in final_state.items():

    print(f"\n[bold cyan]{key}[/bold cyan]")

    print(value)

/Users/riyaz/Personal/Learning/AI/RAG/rag-tutorial-1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


============================================================

SUPERVISOR

============================================================

Next Step: research

...

============================================================

============================================================

SUPERVISOR

============================================================

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ks0f140eerxtbwa95ddp1n5r` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99794, Requested 410. Please try again in 2m56.256s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}